<a href="https://colab.research.google.com/github/mlbl4/machine_learning_techniques/blob/main/Mar%C3%ADa_L%C3%B3pez_Badri_3_4_Supervised_Learning_NaiveBayes_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.4 Supervised Learning: Naive Bayes

- **Instructor**: Juan Ramón Rico
- **Student**: `[María López Badri]`
- `[TODO:]` Correct Colab submission to **be graded**:
  1. **Share this Colab** with `[juanramonua@gmail.com]`.
  2. Use `Practice / Form to send activity (URL Colab - Notebook)` in **Moodle-UA**.

## Summary
---

The basic principles of Bayesian algorithms, specifically the one known as Naive Bayes, will be presented.

Some of these examples are based on the examples from https://www.geeksforgeeks.org/naive-bayes-classifiers/

In [50]:
import numpy as np
import pandas as pd

# Example of probability calculation with Naive Bayes

Bayes' Theorem:

- The probability of an event $A$ conditioned on another event $B$ can be calculated as: $$P(A|B) = \frac{P(A \cap B)}{P(B)} = \frac{P(B|A) P(A)}{P(B)}$$ where $P(A \cap B)$ is the probability of $A$ and $B$ occurring together, and $P(B)$ denotes the probability of $B$.

- In classification, we assume independence between variables, and with normalization, the formula is: $$P(y | x_1, x_2,\dots,x_n) = \frac{P(y) \prod_{n=1}^{n} P(x_i|y)}{\sum_j P(y_j) \prod_{n=1}^{n} P(x_i|y_j)}$$ where $y$ is a class of the target variable, $P(y)$ is the probability of the class $y$, and $P(x_i|y)$ is the conditional probability of $x_i$ given $y$. The class would be the one that achieves the highest probability: $$y = argmax_{y_j} \; P(y_j | x_1, x_2,\dots,x_n)$$

In [51]:
## Data

path = 'https://www.dlsi.ua.es/~juanra/UA/datasets'
path_tennis = f'{path}/tennis-en.csv'
path_covid19 = f'{path}/covid19-en.csv'

data = pd.read_csv(path_tennis)
data

,weather,temperature,humidity,wind,play
0,sunny,high,high,no,no
1,sunny,high,high,yes,no
2,overcast,high,high,no,yes
3,rain,medium,high,no,yes
4,rain,low,normal,no,yes
5,rain,low,normal,yes,no
6,overcast,low,normal,yes,yes
7,sunny,medium,high,no,no
8,sunny,low,normal,no,yes
9,rain,medium,normal,no,yes


# Example 1: TENNIS

## Step 1

- In the Naive version, Bayes' theorem is applied considering the input variables as independent to simplify the calculations.
- Basically, we will create a system to count the repetitions of a category of an input variable with respect to the target.
- With the previous repetitions, Bayes' theorem will be applied to calculate the conditional probability of belonging to the classes of the target variable and thus classify a given input.

In [52]:
y_name = data.columns[-1]     # The name of the last column
X_names = data.columns[:-1]   # The name of all variables up to the last column (excluded)

y_classes = data[y_name].unique() # The classes that need to be predicted
#.unique() removes repetitions
N = len(data)  #how many rows                    # Number of samples

print('target', y_name, y_classes)
print('Input variables', X_names)
print('Number of examples', N)

target play ['no' 'yes']
Input variables Index(['weather', 'temperature', 'humidity', 'wind'], dtype='object')
Number of examples 14


### Prior Probability

What is the probability of guessing correctly without additional information?
We need to count how many times the classes of the target variable are repeated and normalize.

In [53]:
df_count_y = pd.DataFrame(data[y_name].value_counts()) #.value_counts() --> different values
df_prob_y = (df_count_y / df_count_y.sum()).rename(columns={'count':'prob'}).T

print('df_count_y')
display(df_count_y)

print('\ndf_prob_y')
display(df_prob_y)

df_count_y


,count
play,
yes,9
no,5



df_prob_y


play,yes,no
prob,0.642857,0.357143


## Step 2

Suppose that the input $X$ is formed only by the first variable. We would calculate the probabilities as follows:

In [54]:
x_name = X_names[0] # 'Weather' or 'Fever' depending on the chosen dataset (input variables).

# Count the number of times the feature category is repeated according to the target category
# unstack() moves the row values (counts) into new columns
# fillna(0.0) prevents null values when there are no repetitions in a category
df_count = data[[x_name,y_name]].value_counts().unstack().fillna(0.0)
df_prob = df_count / df_count.sum()

print('Count')
display(df_count)

print('\nProb')
display(df_prob.round(2))

Count


play,no,yes
weather,,
overcast,0.0,4.0
rain,2.0,3.0
sunny,3.0,2.0



Prob


play,no,yes
weather,,
overcast,0.0,0.44
rain,0.4,0.33
sunny,0.6,0.22


Suppose the case where $X=(rain)$ to calculate the probability of playing or not would be to calculate $$P(\textrm{play}=\textrm{yes}|\textrm{weather}=\textrm{rain})$$ and $$P(\textrm{play}=\textrm{no}|\textrm{weather}=\textrm{rain})$$

Let us remember the formula: $$P(y | x_1, x_2,\dots,x_n) = \frac{P(y) \prod_{n=1}^{n} P(x_i|y)}{\sum_j P(y_j) \prod_{n=1}^{n} P(x_i|y_j)}$$

In [55]:
df_prob.columns, df_prob.index

(Index(['no', 'yes'], dtype='object', name='play'),
 Index(['overcast', 'rain', 'sunny'], dtype='object', name='weather'))

In [56]:
(df_prob_y['yes']*df_prob.loc['rain','yes']).values[0] #df_prob.loc['rain','yes'] picks the cell in row, and column

np.float64(0.2142857142857143)

In [57]:
# As example, we compute the following case:

# P(play=yes|weather=rain)
numerator_yes = (df_prob_y['yes']*df_prob.loc['rain','yes']).values[0]

# P(play=no|weather=rain)
numerator_no = (df_prob_y['no']*df_prob.loc['rain','no']).values[0]

P_yes = numerator_yes / (numerator_yes + numerator_no)
P_no = numerator_no / (numerator_yes + numerator_no)

print(f'P(play=yes|weather=rain)={P_yes:.4f}')
print(f'P(play=no|weather=rain)={P_no:.4f}')

P(play=yes|weather=rain)=0.6000
P(play=no|weather=rain)=0.4000


- Calculate the conditional repetitions of the categories of each variable with respect to the classes of the target variable.

In [58]:


# Count the number of times the feature category is repeated according to the target category
  # unstack() moves the row values (counts) into new columns
  # fillna(0.0) prevents null values when there are no repetitions in a category

  # TODO: As in the previous example

  # Transform the number of times into probabilities according to the target categories

  # TODO: Add the necessary code

dict_df_prob = {}
X_names = data.columns[:-1]   # The name of all variables up to the last column (excluded)

for X_name in X_names:
  df_count = data[[X_name, y_name]].value_counts().unstack().fillna(0.0)
  dict_df_prob[X_name] = df_count / df_count.sum() # prob for a
  display(dict_df_prob[X_name].round(2))
  print()



play,no,yes
weather,,
overcast,0.0,0.44
rain,0.4,0.33
sunny,0.6,0.22


play,no,yes
temperature,,
high,0.4,0.22
low,0.2,0.33
medium,0.4,0.44


play,no,yes
humidity,,
high,0.8,0.33
normal,0.2,0.67


play,no,yes
wind,,
no,0.4,0.67
yes,0.6,0.33


## Step 3

Calculate the probabilities of the following inputs

In [59]:
X_test_tennis = pd.DataFrame([
    ['overcast', 'low',    'normal', 'yes'],
    ['rain',     'medium', 'normal', 'no' ],
    ['sunny',    'high',   'normal', 'no' ]
  ]
)

X_test_covid19 = pd.DataFrame([
    ['Yes', 'Yes', 'Yes'],
    ['No',  'Yes', 'No' ],
    ['Yes', 'Yes', 'No' ]
  ]
)
X_test = X_test_tennis
X_test.columns = X_names
X_test

,weather,temperature,humidity,wind
0,overcast,low,normal,yes
1,rain,medium,normal,no
2,sunny,high,normal,no


In [60]:
df_res = X_test.copy()
df_res[y_classes] = 0.0
for idx, x in X_test.iterrows():
  for y in y_classes:
    print(y)
    prior = df_prob_y[y].values[0] #value[0] because numpy returns an array
    probs = np.array([dict_df_prob[name_x].loc[x[name_x], y] for name_x in X_names])
    df_res.loc[idx, y] = prior * probs.prod()
    #TODO : Add the necessary code to perform the calculations (counts)
    print(' ', idx, y, round(prior, 4), probs.round(4))

# Display the values
#display(df_res)

# Warning: Now to compute the probabilities the values need to be normalized
df_res[y_classes] = df_res[y_classes].apply(lambda x: x/x.sum(), axis=1).round(4)
df_res

no
  0 no 0.3571 [0.  0.2 0.2 0.6]
yes
  0 yes 0.6429 [0.4444 0.3333 0.6667 0.3333]
no
  1 no 0.3571 [0.4 0.4 0.2 0.4]
yes
  1 yes 0.6429 [0.3333 0.4444 0.6667 0.6667]
no
  2 no 0.3571 [0.6 0.4 0.2 0.4]
yes
  2 yes 0.6429 [0.2222 0.2222 0.6667 0.6667]


,weather,temperature,humidity,wind,no,yes
0,overcast,low,normal,yes,0.0000,1.0000
1,rain,medium,normal,no,0.0975,0.9025
2,sunny,high,normal,no,0.3271,0.6729


Results 🏆

```
X_test 0:
  no
  0 no 0.3571 [0.  0.2 0.2 0.6]
  yes
  0 yes 0.6429 [0.4444 0.3333 0.6667 0.3333]

X_test 1:
  no
  1 no 0.3571 [0.4 0.4 0.2 0.4]
  yes
  1 yes 0.6429 [0.3333 0.4444 0.6667 0.6667]

X_test 2:
  no
  2 no 0.3571 [0.6 0.4 0.2 0.4]
  yes
  2 yes 0.6429 [0.2222 0.2222 0.6667 0.6667]

---------------------------------------------------------------------
   weather     temperature     humidity     wind     no         yes
---------------------------------------------------------------------
0  overcast     low            normal       yes     0.0000     1.0000
1  rain         medium         normal       no      0.0975     0.9025
2  sunny        high           normal       no      0.3271     0.6729
---------------------------------------------------------------------
```

Naive Bayes in `scikit-learn (GaussianNB)`

```
-----------------------------------------------------
    weather temperature humidity wind      no     yes
-----------------------------------------------------
0  overcast         low   normal  yes  0.0025  0.9975
1      rain      medium   normal   no  0.0386  0.9614
2     sunny        high   normal   no  0.2875  0.7125
-----------------------------------------------------
```

## Naive Bayes (scikit-learn)

We will obtain the result using `scikit-learn` by training a system with Naive Bayes and showing the results.

In [61]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder

'''
  We need to change the categories of the variables to integer numbers
  We will use a preprocessing function from sklearn called LabelEncoder
'''
dict_label_encoder = {} #dictionary to save the encoder used for each column
data_codes = data.copy()
for i in data.columns:
  le = LabelEncoder()
  le.fit(sorted(data[i].unique()))
  data_codes[i] = le.transform(data[i]) #takes every value in the current column and replaces it with its corresponding numerical code.
  dict_label_encoder[i] = le

model = GaussianNB() # TODO 1: Model
model.fit(data_codes[X_names], data_codes[y_name]) # TODO 2: Train with the corresponding data
# TODO 3: Make predictions on X_test
X_test_codes = X_test.copy()

for i in X_names:
  X_test_codes[i] = dict_label_encoder[i].transform(X_test[i])

df_sk = X_test.copy()
df_sk[dict_label_encoder[y_name].classes_] = model.predict_proba(X_test_codes).round(4)


# Example 2: COVID-19

Apply the three previous steps and `scikit-learn` solution to the COVID-19 dataset

In [62]:
data = pd.read_csv(path_covid19)
x_names = data.columns[:-1]
y_name = 'Infected'

y_classes = data[y_name].unique()
N = len(data) #n of rows

print('target', y_name, y_classes)
print('input vars', x_names)
print('numbers of examples', N)

target Infected ['No' 'Yes']
input vars Index(['Id', 'Fever', 'Cough', 'Respiratory Problems'], dtype='object')
numbers of examples 14


In [63]:
df_count_y = pd.DataFrame(data[y_name].value_counts())
df_prob_y = (df_count_y / df_count_y.sum()).rename(columns={'count':'prob'}).T

print('df_count_y')
display(df_count_y)
print()
print('df_prob_y')
display(df_prob_y)

df_count_y


,count
Infected,
Yes,8
No,6



df_prob_y


Infected,Yes,No
prob,0.571429,0.428571


In [64]:
#We calculate the probability of each category according to the target categorie



#FIRST CALULATE number of times that appear each categorie
dict_prob = {}
for x_name in x_names:
  df_count = data[[x_name, y_name]].value_counts().unstack().fillna(0.0)
  dict_prob[x_name] = df_count / df_count.sum()
  display(dict_df_prob[X_name].round(2))
  print()

  X_test_covid19 = pd.DataFrame([
    ['Yes', 'Yes', 'Yes'],
    ['No',  'Yes', 'No' ],
    ['Yes', 'Yes', 'No' ]
  ]
)

play,no,yes
wind,,
no,0.4,0.67
yes,0.6,0.33


play,no,yes
wind,,
no,0.4,0.67
yes,0.6,0.33


play,no,yes
wind,,
no,0.4,0.67
yes,0.6,0.33


play,no,yes
wind,,
no,0.4,0.67
yes,0.6,0.33


In [65]:
# Asignamos las variables de entrada correctas de COVID-19 excluyendo el 'Id'
x_names_covid = [col for col in x_names if col != 'Id']

# Configuramos X_test_covid19 con los nombres de columna adecuados
X_test_covid19.columns = x_names_covid
X_test = X_test_covid19

df_res = X_test.copy()
df_res[y_classes] = 0.0
for idx, x in X_test.iterrows():
  for y in y_classes:
    print(y)
    prior = df_prob_y[y].values[0] #value[0] because numpy returns an array
    # Usamos x_names_covid en lugar de X_names de tenis
    probs = np.array([dict_prob[name_x].loc[x[name_x], y] for name_x in x_names_covid])
    df_res.loc[idx, y] = prior * probs.prod()
    #TODO : Add the necessary code to perform the calculations (counts)
    print(' ', idx, y, round(prior, 4), probs.round(4))

# Display the values
#display(df_res)

# Warning: Now to compute the probabilities the values need to be normalized
df_res[y_classes] = df_res[y_classes].apply(lambda x: x/x.sum(), axis=1).round(4)
df_res

No
  0 No 0.4286 [0.3333 0.8333 0.1667]
Yes
  0 Yes 0.5714 [0.75  0.625 0.875]
No
  1 No 0.4286 [0.6667 0.8333 0.8333]
Yes
  1 Yes 0.5714 [0.25  0.625 0.125]
No
  2 No 0.4286 [0.3333 0.8333 0.8333]
Yes
  2 Yes 0.5714 [0.75  0.625 0.125]


,Fever,Cough,Respiratory Problems,No,Yes
0,Yes,Yes,Yes,0.0780,0.9220
1,No,Yes,No,0.9467,0.0533
2,Yes,Yes,No,0.7477,0.2523


In [66]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder

'''
  We need to change the categories of the variables to integer numbers
  We will use a preprocessing function from sklearn called LabelEncoder
'''
dict_label_encoder = {}
data_codes = data.copy()
for i in data.columns:
  le = LabelEncoder()
  le.fit(sorted(data[i].unique()))
  data_codes[i] = le.transform(data[i])
  dict_label_encoder[i] = le

# Excluimos 'Id' de las variables de entrenamiento para que coincidan con X_test
x_names_model = [col for col in x_names if col != 'Id']

model = GaussianNB() # TODO 1: Model
model.fit(data_codes[x_names_model], data_codes[y_name]) # TODO 2: Train with the corresponding data
# TODO 3: Make predictions on X_test
X_test_codes = X_test.copy()

for i in x_names_model:
  X_test_codes[i] = dict_label_encoder[i].transform(X_test[i])

df_sk = X_test.copy()
df_sk[dict_label_encoder[y_name].classes_] = model.predict_proba(X_test_codes[x_names_model]).round(4)
df_sk

,Fever,Cough,Respiratory Problems,No,Yes
0,Yes,Yes,Yes,0.0358,0.9642
1,No,Yes,No,0.9902,0.0098
2,Yes,Yes,No,0.9266,0.0734


In [67]:
#sklearn



Results 🏆

```
X_text 0:
	 No
	 0 No 0.4286 [0.3333 0.8333 0.1667]
	 Yes
	 0 Yes 0.5714 [0.75  0.625 0.875]
X_text 1:
	 No
	 1 No 0.4286 [0.6667 0.8333 0.8333]
	 Yes
	 1 Yes 0.5714 [0.25  0.625 0.125]
X_text 2:
	 No
	 2 No 0.4286 [0.3333 0.8333 0.8333]
	 Yes
	 2 Yes 0.5714 [0.75  0.625 0.125]

--------------------------------------------------
  Fever Cough Respiratory Problems      No     Yes
--------------------------------------------------  
0   Yes   Yes                  Yes  0.0780  0.9220
1    No   Yes                   No  0.9467  0.0533
2   Yes   Yes                   No  0.7477  0.2523
--------------------------------------------------
```

Naive Bayes in `scikit-learn (GaussianNB)`

```
--------------------------------------------------
  Fever Cough Respiratory Problems      No     Yes
--------------------------------------------------
0   Yes   Yes                  Yes  0.0358  0.9642
1    No   Yes                   No  0.9902  0.0098
2   Yes   Yes                   No  0.9266  0.0734
--------------------------------------------------
```


# AI-Assisted Code Optimization Exercise

Use a generative AI tool to optimize the implementation of the previous **Naïve Bayes** related functions.

* **Optimize the code:** Use a generative AI tool to improve the efficiency of the original implementation.
* **Explain the changes:** Briefly describe the main changes introduced by the AI-generated version and explain how they improve the computational efficiency of the original code.
* **Validate the result:** Test the optimized implementation using one of the examples from the notebook and verify that it produces results equivalent to those of the original implementation.


In [68]:
def naive_bayes_opt(data, X_test):
  y_name = data.columns[-1]
  X_names = list(data.columns[:-1])

  # Encode the target as integers once
  y_codes, y_classes = pd.factorize(data[y_name])
  n_y = len(y_classes)

  # Log of the prior, one copy per test sample
  prior = np.bincount(y_codes, minlength=n_y) / len(data)
  log_post = np.tile(np.log(prior), (len(X_test), 1))

  with np.errstate(divide='ignore'):          # log(0) = -inf is expected for zero counts
    for name in X_names:
      x_codes, cats = pd.factorize(data[name])
      counts = np.zeros((len(cats), n_y))
      np.add.at(counts, (x_codes, y_codes), 1)          # contingency table in one operation
      log_cond = np.log(counts / counts.sum(axis=0))    # log P(x | y)
      t = cats.get_indexer(X_test[name])                # test values → row numbers
      log_post += np.where(t[:, None] >= 0, log_cond[t], -np.inf)   # all test samples at once

  # Normalize (stable version of x / x.sum())
  post = np.exp(log_post - log_post.max(axis=1, keepdims=True))
  post /= post.sum(axis=1, keepdims=True)

  res = X_test.copy()
  res[list(y_classes)] = post.round(4)
  return res


The optimized version encodes each column as integers and builds each conditional probability table with a single NumPy operation (np.add.at), instead of using value_counts, unstack and fillna with pandas. The main improvement is in the prediction: the original loops over every test sample and every class with iterrows, and looks up each probability with .loc, which is slow. The optimized version computes all test samples and all classes at once with array indexing. It also works with logarithms, adding log-probabilities instead of multiplying probabilities; this gives the same result but avoids numbers becoming so small that they round to 0 when there are many features. Finally, the normalization is done with NumPy instead of apply with a lambda.

In [70]:
df_opt = naive_bayes_opt(data.drop(columns='Id'), X_test)
display(df_opt)
print('Same results:', np.allclose(df_res[y_classes], df_opt[y_classes]))

,Fever,Cough,Respiratory Problems,No,Yes
0,Yes,Yes,Yes,0.0780,0.9220
1,No,Yes,No,0.9467,0.0533
2,Yes,Yes,No,0.7477,0.2523


Same results: True
